# Chapter 4.3: Sampling Depth and Claim Boundaries

This notebook isolates the Chapter 4 sampling-depth, WFR-FM sensitivity, stochastic bridge, and claim-boundary audit experiments. The prior-boundary component is a table audit, not a real-data prior experiment.


## 0. Setup

Imports, paths, device selection, random seeds, output directories, plotting style, cache helpers, and run-size controls.  The defaults below match the requested chapter settings; environment variables are only for smoke testing or resuming partial runs.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_ch04")

from pathlib import Path
import sys
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    from torch import nn
except Exception as exc:
    raise ImportError("Chapter 4 experiments require PyTorch.") from exc

try:
    import anndata as ad
except Exception:
    ad = None

from scipy import sparse
from scipy.sparse.csgraph import shortest_path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/home/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import VelocityMLP as ProjectVelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample
from src.samplers import CouplingPairSampler as SrcCouplingPairSampler
from src.ot import (
    pairwise_squared_distances,
    independent_coupling,
    compute_ot_coupling_from_cost,
    sample_pair_indices_from_coupling,
    coupling_diagnostics,
)
from src.metrics import (
    mmd_rbf as project_mmd_rbf,
    sliced_wasserstein_distance,
    endpoint_metrics,
    trajectory_path_length,
    trajectory_path_energy,
    trajectory_straightness,
    off_manifold_knn_distance,
    fate_mass_error,
    coupling_l1_distance,
    normalized_cost_matrix,
    distribution_readout_metrics,
)
from src.representations import (
    fit_pca_state_space,
    pca_inverse_transform,
    program_index_dict,
    readout_program_scores_from_matrix,
    standardize_train_space,
    nearest_neighbor_overlap,
)

SEEDS = [42, 43, 44]
DEFAULT_SEED = 42
SOURCE_TIME = "1"
TARGET_TIME = "2"
TRAINING_STEPS = int(os.environ.get("CH04_TRAINING_STEPS", "1500"))
BATCH_SIZE = int(os.environ.get("CH04_BATCH_SIZE", "256"))
DEFAULT_NFE = int(os.environ.get("CH04_DEFAULT_NFE", "64"))
NFE_GRID = [2, 4, 8, 16, 32, 64]
SINKHORN_EPSILON = float(os.environ.get("CH04_SINKHORN_EPSILON", "0.05"))
EPSILON_GRID = [0.01, 0.02, 0.05, 0.1, 0.5]
BOOTSTRAP_REPEATS = int(os.environ.get("CH04_BOOTSTRAP_REPEATS", "50"))
EB_MAX_CELLS_PER_TIME = os.environ.get("CH04_EB_MAX_CELLS_PER_TIME", "")
EB_MAX_CELLS_PER_TIME = None if EB_MAX_CELLS_PER_TIME == "" else int(EB_MAX_CELLS_PER_TIME)
TOY_TRAINING_STEPS = int(os.environ.get("CH04_TOY_TRAINING_STEPS", str(TRAINING_STEPS)))
SMOKE_MODE = os.environ.get("CH04_SMOKE_MODE", "0") == "1"
if SMOKE_MODE:
    TRAINING_STEPS = min(TRAINING_STEPS, 20)
    TOY_TRAINING_STEPS = min(TOY_TRAINING_STEPS, 20)
    BATCH_SIZE = min(BATCH_SIZE, 64)
    DEFAULT_NFE = min(DEFAULT_NFE, 8)
    NFE_GRID = [2, 4, 8]
    BOOTSTRAP_REPEATS = min(BOOTSTRAP_REPEATS, 3)
    EB_MAX_CELLS_PER_TIME = 120 if EB_MAX_CELLS_PER_TIME is None else min(EB_MAX_CELLS_PER_TIME, 120)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = PROJECT_ROOT / "data"
EB_PATH = DATA_DIR / "trajectorynet_eb" / "eb_velocity_v5.npz"
TOY_DIR = DATA_DIR / "toy_branching_snapshots"
TOY_CSV_PATH = TOY_DIR / "observed_2d_snapshots.csv"
TOY_H5AD_PATH = TOY_DIR / "branching_toy_pseudocounts.h5ad"

FIG_DIR = PROJECT_ROOT /  "figures" / "ch04"
OUT_DIR = PROJECT_ROOT /  "outputs" / "ch04"
if SMOKE_MODE:
    FIG_DIR = FIG_DIR / "smoke"
    OUT_DIR = OUT_DIR / "smoke"
CACHE_DIR = OUT_DIR / "cache"
for path in [FIG_DIR, OUT_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = {
    "source": "#4C78A8",
    "target": "#F58518",
    "random": "#8E8E8E",
    "ot": "#008A7A",
    "reflow1": "#5369A6",
    "reflow2": "#B279A2",
    "rare": "#D95F02",
    "major": "#2C7FB8",
    "program": "#54A24B",
    "diagnostic": "#E45756",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Training steps: {TRAINING_STEPS}; batch size: {BATCH_SIZE}; default NFE: {DEFAULT_NFE}")
print(f"Smoke mode: {SMOKE_MODE}")


In [ ]:
def set_seed(seed: int = DEFAULT_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_ready(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    return obj


def save_json(path: str | Path, payload: dict) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True))
    return path


def load_json(path: str | Path):
    return json.loads(Path(path).read_text())


def save_npz(path: str | Path, **arrays) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)
    return path


def load_npz(path: str | Path):
    return np.load(Path(path), allow_pickle=True)


def save_csv(path: str | Path, frame: pd.DataFrame) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path


def save_pt(path: str | Path, payload) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)
    return path


def load_pt(path: str | Path, map_location=None):
    return torch.load(Path(path), map_location=map_location or DEVICE)


def save_figure(fig, filename: str, close: bool = True) -> Path:
    path = FIG_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    if close:
        plt.close(fig)
    return path


def artifact_exists(*paths) -> bool:
    return all(Path(p).exists() and Path(p).stat().st_size > 0 for p in paths)


def stable_hash(*items) -> str:
    h = hashlib.sha1()
    for item in items:
        h.update(str(item).encode())
    return h.hexdigest()[:10]


def sample_rows(n: int, max_n: int | None, seed: int = DEFAULT_SEED) -> np.ndarray:
    idx = np.arange(int(n))
    if max_n is None or n <= max_n:
        return idx
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(idx, size=int(max_n), replace=False))


def as_float32(x):
    return np.asarray(x, dtype=np.float32)


def to_tensor(x, device: torch.device = DEVICE):
    return torch.as_tensor(x, dtype=torch.float32, device=device)


def ensure_finite(name: str, x: np.ndarray) -> None:
    if not np.all(np.isfinite(x)):
        raise ValueError(f"{name} contains non-finite values")


## 1. Shared Utilities

The cells below define chapter-level wrappers and helpers.  They intentionally keep the key CFM/Sinkhorn/rollout/metric logic visible while reusing tested primitives from `src` where available.


In [ ]:
class VelocityMLP(ProjectVelocityMLP):
    """Chapter 4 VelocityMLP wrapper with requested defaults.

    The underlying implementation is imported from `src.models`, but the
    requested architecture is fixed here: hidden=128 and layers=4.
    """
    def __init__(self, input_dim: int, hidden: int = 128, layers: int = 4):
        super().__init__(x_dim=int(input_dim), hidden_dim=int(hidden), hidden_layers=int(layers))


def make_time_batch(batch_size: int, device: torch.device = DEVICE) -> torch.Tensor:
    return torch.rand(int(batch_size), 1, device=device)


def sample_independent_pairs(X0, X1, n_pairs: int, seed: int = DEFAULT_SEED):
    rng = np.random.default_rng(seed)
    i0 = rng.integers(0, len(X0), size=int(n_pairs))
    i1 = rng.integers(0, len(X1), size=int(n_pairs))
    return i0, i1


def compute_cost_matrix(x0, x1, normalize: bool = True):
    C = pairwise_squared_distances(np.asarray(x0, dtype=np.float32), np.asarray(x1, dtype=np.float32)).astype(np.float32)
    if not normalize:
        return C, 1.0
    positive = C[C > 0]
    scale = float(np.median(positive)) if positive.size else 1.0
    scale = max(scale, 1e-12)
    return (C / scale).astype(np.float32), scale


def sinkhorn_plan(C, epsilon: float = SINKHORN_EPSILON, return_info: bool = False):
    """Balanced Sinkhorn plan using POT when available, with project fallback."""
    return compute_ot_coupling_from_cost(np.asarray(C, dtype=np.float32), epsilon=float(epsilon), return_info=return_info)


def sample_from_plan(pi, n_pairs: int, seed: int = DEFAULT_SEED):
    return sample_pair_indices_from_coupling(np.asarray(pi, dtype=float), batch_size=int(n_pairs), seed=seed)


class PlanPairSampler:
    """Notebook-local thin wrapper around `src.samplers.CouplingPairSampler`."""
    def __init__(self, X0, X1, pi=None, seed: int = DEFAULT_SEED, labels0=None, labels1=None):
        self.X0 = as_float32(X0)
        self.X1 = as_float32(X1)
        if pi is None:
            pi = independent_coupling(len(self.X0), len(self.X1))
        self.pi = np.asarray(pi, dtype=float)
        self.src_sampler = SrcCouplingPairSampler(self.X0, self.X1, self.pi, seed=seed, labels0=labels0, labels1=labels1)

    def sample(self, batch_size: int = BATCH_SIZE):
        return self.src_sampler.sample(int(batch_size))


def train_cfm(
    model,
    pair_sampler,
    steps: int = TRAINING_STEPS,
    batch_size: int = BATCH_SIZE,
    lr: float = 1e-3,
    device: torch.device = DEVICE,
    seed: int = DEFAULT_SEED,
    log_every: int = 250,
):
    """Thin notebook wrapper around `src.train.train_cfm_steps`."""
    set_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(lr))
    history = train_cfm_steps(
        model,
        pair_sampler.sample,
        optimizer,
        n_steps=int(steps),
        batch_size=int(batch_size),
        device=device,
        log_every=int(log_every),
    )
    return history


@torch.no_grad()
def rollout_euler(model, x0, nfe: int = DEFAULT_NFE, device: torch.device = DEVICE):
    """Thin notebook wrapper around `src.sampling.euler_sample`."""
    model.eval()
    x0_t = to_tensor(x0, device)
    endpoint, _, _ = euler_sample(model, x0_t, n_steps=int(nfe), return_traj=False)
    return endpoint.detach().cpu().numpy().astype(np.float32)


@torch.no_grad()
def trajectory_rollout(model, x0, nfe: int = DEFAULT_NFE, return_path: bool = True, device: torch.device = DEVICE):
    """Thin notebook wrapper around `src.sampling.euler_sample`."""
    model.eval()
    x0_t = to_tensor(x0, device)
    endpoint, traj_t, _ = euler_sample(model, x0_t, n_steps=int(nfe), return_traj=return_path)
    endpoint_np = endpoint.detach().cpu().numpy().astype(np.float32)
    if return_path:
        traj_np = traj_t.detach().cpu().numpy().astype(np.float32)
        return endpoint_np, traj_np, np.linspace(0.0, 1.0, int(nfe) + 1)
    return endpoint_np


def mmd_rbf(X, Y, gamma: float | None = None):
    return project_mmd_rbf(np.asarray(X, dtype=float), np.asarray(Y, dtype=float), gamma=gamma)


def sliced_w2(X, Y, n_projections: int = 128, seed: int = DEFAULT_SEED):
    return sliced_wasserstein_distance(X, Y, n_projections=n_projections, seed=seed)


def path_length(traj):
    return trajectory_path_length(np.asarray(traj, dtype=float))


def path_energy(traj, times=None):
    return trajectory_path_energy(np.asarray(traj, dtype=float), times=times)


def tortuosity_straightness(traj):
    """Path-length/end-point tortuosity minus one; geometric, not biological validation."""
    return trajectory_straightness(np.asarray(traj, dtype=float))


def straightness(traj):
    """Backward-compatible alias used only inside this notebook."""
    return tortuosity_straightness(traj)


def straightness_action_S(traj, times=None):
    """Planning straightness action S = integral E[||(Z1-Z0)-dZ_t/dt||^2] dt.

    Euler trajectories provide finite-difference velocities. Lower S means
    finite-difference path velocities are closer to the endpoint chord.
    """
    traj = np.asarray(traj, dtype=float)
    if traj.ndim != 3 or traj.shape[0] < 2:
        raise ValueError("traj must have shape (T, N, D) with T >= 2")
    if times is None:
        times = np.linspace(0.0, 1.0, traj.shape[0])
    times = np.asarray(times, dtype=float)
    dt = np.diff(times)
    if len(dt) != traj.shape[0] - 1 or np.any(dt <= 0):
        raise ValueError("times must be strictly increasing and match traj")
    vel = np.diff(traj, axis=0) / dt[:, None, None]
    chord = traj[-1] - traj[0]
    sq = np.sum((chord[None, :, :] - vel) ** 2, axis=-1)
    return float(np.sum(sq.mean(axis=1) * dt))


def off_manifold_knn(points, reference, k: int = 15, batch_size: int = 1000):
    points_arr = np.asarray(points, dtype=np.float32)
    reference_arr = np.asarray(reference, dtype=np.float32)
    if points_arr.ndim == 3:
        points_arr = points_arr.reshape(-1, points_arr.shape[-1])
    if points_arr.ndim != 2 or reference_arr.ndim != 2:
        raise ValueError("points and reference must be 2D arrays, or points may be a (T, N, D) trajectory")
    if points_arr.shape[1] != reference_arr.shape[1]:
        raise ValueError("points and reference must have the same feature dimension")
    if points_arr.shape[0] <= batch_size:
        return off_manifold_knn_distance(points_arr, reference_arr, k=k)
    from sklearn.neighbors import NearestNeighbors
    kk = max(1, min(int(k), reference_arr.shape[0]))
    nn = NearestNeighbors(n_neighbors=kk, algorithm="ball_tree", leaf_size=40, n_jobs=1)
    nn.fit(reference_arr)
    total = 0.0
    count = 0
    for start in range(0, points_arr.shape[0], int(batch_size)):
        distances, _ = nn.kneighbors(points_arr[start:start + int(batch_size)])
        total += float(distances.sum())
        count += int(distances.size)
    return total / max(count, 1)

def coarse_step_error(model, x0, nfe_coarse: int = 4, nfe_fine: int = 64):
    z_coarse = rollout_euler(model, x0, nfe=nfe_coarse)
    z_fine = rollout_euler(model, x0, nfe=nfe_fine)
    return float(np.linalg.norm(z_coarse - z_fine, axis=1).mean())


def effective_support(pi):
    pi = np.asarray(pi, dtype=float)
    p = pi[pi > 0]
    return float(np.exp(-np.sum(p * np.log(p)))) if p.size else 0.0


def plan_entropy(pi):
    pi = np.asarray(pi, dtype=float)
    p = pi[pi > 0]
    return -float(np.sum(p * np.log(p))) if p.size else 0.0


def topk_nn_overlap(X_a, X_b, k: int = 15):
    return nearest_neighbor_overlap(X_a, X_b, k=k)


def coupling_topk_overlap(pi_a, pi_b, k: int = 15):
    pi_a = np.asarray(pi_a, dtype=float)
    pi_b = np.asarray(pi_b, dtype=float)
    if pi_a.shape != pi_b.shape:
        raise ValueError("coupling shapes differ")
    k = max(1, min(int(k), pi_a.shape[1]))
    rows = []
    for a, b in zip(pi_a, pi_b):
        ta = set(np.argpartition(-a, kth=k - 1)[:k].tolist())
        tb = set(np.argpartition(-b, kth=k - 1)[:k].tolist())
        rows.append(len(ta & tb) / float(k))
    return float(np.mean(rows))


def evaluate_endpoint(pred, target, seed: int = DEFAULT_SEED):
    return {
        "endpoint_mmd": mmd_rbf(pred, target),
        "sliced_w2": sliced_w2(pred, target, seed=seed),
    }


In [ ]:
def plot_phate_pairs(X0_phate, X1_phate, idx0, idx1, title: str, max_lines: int = 120, seed: int = DEFAULT_SEED, values=None):
    return plot_pair_panels(
        X0_phate,
        X1_phate,
        [{"title": title, "idx0": idx0, "idx1": idx1, "values": values, "seed": seed}],
        filename="_temporary_pair_panel.png",
        title=title,
    )


def plot_pair_panels(X0_phate, X1_phate, panels, filename: str, title: str, value_label: str = "PC-20 chord length"):
    from matplotlib.collections import LineCollection
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    axes_flat = axes[0]
    all_values = []
    for panel in panels:
        if panel.get("values") is not None:
            all_values.append(np.asarray(panel["values"], dtype=float))
    norm = None
    if all_values:
        finite = np.concatenate([v[np.isfinite(v)] for v in all_values if np.any(np.isfinite(v))])
        norm = Normalize(vmin=float(finite.min()), vmax=float(finite.max())) if finite.size else None
    for ax, panel in zip(axes_flat, panels):
        idx0, idx1 = panel["idx0"], panel["idx1"]
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.20, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.20, linewidths=0)
        keep = sample_rows(len(idx0), min(panel.get("max_lines", 100), len(idx0)), seed=panel.get("seed", DEFAULT_SEED))
        segments = np.stack([X0_phate[idx0[keep]], X1_phate[idx1[keep]]], axis=1)
        values = panel.get("values")
        if values is not None and norm is not None:
            lc = LineCollection(segments, cmap="viridis", norm=norm, linewidths=0.8, alpha=0.55)
            lc.set_array(np.asarray(values, dtype=float)[keep])
            ax.add_collection(lc)
        else:
            lc = LineCollection(segments, colors=panel.get("color", "0.45"), linewidths=0.7, alpha=0.25)
            ax.add_collection(lc)
        ax.set_title(panel["title"])
        ax.set_xlabel("PHATE 1")
        ax.set_ylabel("PHATE 2")
    if norm is not None:
        fig.colorbar(ScalarMappable(norm=norm, cmap="viridis"), ax=list(axes_flat), fraction=0.035, pad=0.02, label=value_label)
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


def add_local_arrows(ax, projected_traj, observed_phate, color, max_arrows: int = 28, seed: int = DEFAULT_SEED):
    """Draw local average arrows only for trajectory points close to observed PHATE neighborhoods."""
    projected_traj = np.asarray(projected_traj, dtype=float)
    observed_phate = np.asarray(observed_phate, dtype=float)
    if projected_traj.shape[0] < 3 or projected_traj.shape[-1] != 2:
        return
    mid_step = projected_traj.shape[0] // 2
    anchors = projected_traj[mid_step]
    deltas = projected_traj[min(mid_step + 1, projected_traj.shape[0] - 1)] - projected_traj[max(mid_step - 1, 0)]
    nn_obs = NearestNeighbors(n_neighbors=min(15, len(observed_phate))).fit(observed_phate)
    obs_r = nn_obs.kneighbors(observed_phate, return_distance=True)[0][:, -1]
    dist = nn_obs.kneighbors(anchors, return_distance=True)[0][:, 0]
    threshold = float(np.quantile(obs_r, 0.75))
    valid = np.flatnonzero(dist <= threshold)
    if valid.size == 0:
        return
    keep = sample_rows(len(valid), min(max_arrows, len(valid)), seed=seed)
    valid = valid[keep]
    ax.quiver(
        anchors[valid, 0], anchors[valid, 1], deltas[valid, 0], deltas[valid, 1],
        angles="xy", scale_units="xy", scale=1.0, width=0.004, color=color, alpha=0.75,
    )


def plot_projected_trajectories(paths, X0_phate, X1_phate, pc_to_phate, filename: str, title: str, max_lines: int = 45, local_arrows: bool = True):
    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    observed_phate = np.vstack([X0_phate, X1_phate])
    for ax, (name, traj) in zip(axes[0], paths.items()):
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.18, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.16, linewidths=0)
        keep = sample_rows(traj.shape[1], min(max_lines, traj.shape[1]), seed=DEFAULT_SEED)
        selected = np.asarray(traj[:, keep, :], dtype=np.float32)
        flat = selected.reshape(-1, selected.shape[-1])
        ph = pc_to_phate(flat).reshape(selected.shape[0], selected.shape[1], 2)
        color = PALETTE.get(name, "0.35")
        for j in range(ph.shape[1]):
            ax.plot(ph[:, j, 0], ph[:, j, 1], color=color, alpha=0.55, linewidth=1.0)
        if local_arrows:
            add_local_arrows(ax, ph, observed_phate, color=color, seed=DEFAULT_SEED + 7)
        ax.set_title(name.replace("_", " "))
        ax.set_xlabel("PHATE 1 (display only)")
        ax.set_ylabel("PHATE 2 (display only)")
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


def plot_metric_lines(table, x: str, y: str, hue: str, filename: str, title: str):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    for name, group in table.groupby(hue):
        group = group.sort_values(x)
        ax.plot(group[x], group[y], marker="o", linewidth=1.5, label=str(name).replace("_", " "))
    ax.set_xscale("log", base=2 if set(table[x]).issubset(set(NFE_GRID)) else 10)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.legend(frameon=False)
    return save_figure(fig, filename)


def plot_heatmap(matrix, title: str, filename: str, max_size: int = 120, cmap: str = "viridis"):
    M = np.asarray(matrix)
    rows = sample_rows(M.shape[0], min(max_size, M.shape[0]), seed=DEFAULT_SEED)
    cols = sample_rows(M.shape[1], min(max_size, M.shape[1]), seed=DEFAULT_SEED + 1)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(M[np.ix_(rows, cols)], aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("target subset")
    ax.set_ylabel("source subset")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return save_figure(fig, filename)


def plot_table_image(frame: pd.DataFrame, filename: str, title: str, max_rows: int = 12):
    shown = frame.head(max_rows).copy()
    fig, ax = plt.subplots(figsize=(min(14, 1.5 + 1.4 * shown.shape[1]), 0.8 + 0.35 * len(shown)))
    ax.axis("off")
    ax.set_title(title, loc="left")
    tbl = ax.table(cellText=shown.round(4).astype(str).values, colLabels=shown.columns, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1.0, 1.25)
    return save_figure(fig, filename)


## 2. Load EB Data

EB `pcs` is loaded from `eb_velocity_v5.npz`.  This file stores 100 PCs, so this notebook explicitly takes the first 20 PCs and standardizes them using all EB snapshots.  `phate` is kept as display-only coordinates.  The main EB bridge is time label `1 -> 2`.


In [ ]:
def load_eb_data(path: Path = EB_PATH, source_time: str = SOURCE_TIME, target_time: str = TARGET_TIME):
    if not path.exists():
        raise FileNotFoundError(path)
    z = np.load(path, allow_pickle=True)
    keys = list(z.files)
    pcs_full = np.asarray(z["pcs"], dtype=np.float32)
    phate = np.asarray(z["phate"], dtype=np.float32)[:, :2]
    labels = np.asarray(z["sample_labels"]).astype(str)
    pcs20_raw = pcs_full[:, :20].astype(np.float32)
    mean = pcs20_raw.mean(axis=0)
    std = pcs20_raw.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    pcs20 = ((pcs20_raw - mean) / std).astype(np.float32)
    available = sorted(np.unique(labels).tolist(), key=lambda s: int(s) if str(s).isdigit() else str(s))
    if str(source_time) not in available or str(target_time) not in available:
        raise ValueError(f"Requested bridge {source_time}->{target_time}; available labels: {available}")
    idx_source_all = np.flatnonzero(labels == str(source_time))
    idx_target_all = np.flatnonzero(labels == str(target_time))
    if EB_MAX_CELLS_PER_TIME is not None:
        idx_source = idx_source_all[sample_rows(len(idx_source_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED)]
        idx_target = idx_target_all[sample_rows(len(idx_target_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED + 1)]
    else:
        idx_source, idx_target = idx_source_all, idx_target_all
    counts = pd.Series(labels, name="time").value_counts().sort_index().rename_axis("time").reset_index(name="n_cells")
    summary = {
        "source_path": str(path),
        "available_keys": keys,
        "pcs_shape_actual": list(pcs_full.shape),
        "pc20_shape_used": list(pcs20.shape),
        "phate_shape": list(phate.shape),
        "sample_label_values": counts.to_dict(orient="records"),
        "source_time": str(source_time),
        "target_time": str(target_time),
        "n_source_full": int(len(idx_source_all)),
        "n_target_full": int(len(idx_target_all)),
        "n_source_used": int(len(idx_source)),
        "n_target_used": int(len(idx_target)),
        "training_space": "standardized PC-20 from pcs[:, :20]",
        "ot_cost_space": "standardized PC-20 unless Exp 7 contrastive diagnostic",
        "display_space": "PHATE 2D only for visualization",
        "distributional_metrics_space": "standardized PC-20",
        "standardization": "mean/std fit on all EB snapshots in PC-20",
        "adaptation_note": "Input pcs has 100 columns; this chapter uses the first 20 PCs as PC-20.",
    }
    save_json(OUT_DIR / "eb_data_summary.json", summary)
    return {
        "keys": keys,
        "pcs20_all": pcs20,
        "pcs20_raw_all": pcs20_raw,
        "phate_all": phate,
        "labels": labels,
        "counts": counts,
        "pc_mean": mean,
        "pc_std": std,
        "idx_source": idx_source,
        "idx_target": idx_target,
        "X0_pc": pcs20[idx_source],
        "X1_pc": pcs20[idx_target],
        "X0_phate": phate[idx_source],
        "X1_phate": phate[idx_target],
        "source_time": str(source_time),
        "target_time": str(target_time),
        "summary": summary,
    }

EB = load_eb_data()
print("EB keys:", EB["keys"])
print(EB["counts"])
print("Source/target PC shapes:", EB["X0_pc"].shape, EB["X1_pc"].shape)
print("Summary saved to", OUT_DIR / "eb_data_summary.json")


In [ ]:
# Display-only mapping from PC-20 trajectory points back to PHATE for plotting.
# This is not used for training or endpoint distributional metrics.
pc_to_phate_knn = KNeighborsClassifier(n_neighbors=min(15, len(EB["pcs20_all"])), weights="distance")
pc_to_phate_knn.fit(EB["pcs20_all"], np.arange(len(EB["pcs20_all"])))

def pc_to_phate(points_pc):
    points_pc = np.asarray(points_pc, dtype=np.float32)
    dist, ind = pc_to_phate_knn.kneighbors(points_pc, return_distance=True)
    w = 1.0 / np.clip(dist, 1e-6, None)
    w = w / w.sum(axis=1, keepdims=True)
    return np.einsum("nk,nkd->nd", w, EB["phate_all"][ind])

off_manifold_reference_pc = EB["pcs20_all"]
off_manifold_reference_note = "all available EB snapshots in standardized PC-20"
print(off_manifold_reference_note, off_manifold_reference_pc.shape)


## Exp 9. EB Equal-Depth Subsampling

Use all EB time points.  If no observed state labels are present, create coarse state bins with k-means in standardized PC-20.  Counts are reported as sampling-depth diagnostics and normalized proportions, not absolute biological abundance.


In [ ]:
labels_all = EB["labels"]
pcs_all = EB["pcs20_all"]
unique_times = sorted(np.unique(labels_all).tolist(), key=lambda s: int(s) if str(s).isdigit() else str(s))
# EB npz has no cell-type/state labels; create coarse bins on PC-20.
n_bins = min(8, max(2, int(np.sqrt(len(unique_times) * 8))))
kmeans = KMeans(n_clusters=n_bins, random_state=42, n_init=10)
state_bins = kmeans.fit_predict(pcs_all).astype(str)
all_bins = sorted(np.unique(state_bins).tolist(), key=lambda s: int(s) if str(s).isdigit() else s)

counts_rows = []
for t in unique_times:
    mask = labels_all == t
    total = int(mask.sum())
    count_by_bin = pd.Series(state_bins[mask]).value_counts()
    for b in all_bins:
        c = int(count_by_bin.get(b, 0))
        counts_rows.append({"time": t, "state_bin": b, "n_cells": c, "total_time_cells": total, "proportion": float(c / total)})
counts_by_state = pd.DataFrame(counts_rows)

n_min = int(pd.Series(labels_all).value_counts().min())
rng = np.random.default_rng(303)
equal_selected = []
for t in unique_times:
    idx = np.flatnonzero(labels_all == t)
    equal_selected.append(rng.choice(idx, size=n_min, replace=False))
equal_selected = np.concatenate(equal_selected)
equal_rows = []
for t in unique_times:
    idx_t = equal_selected[labels_all[equal_selected] == t]
    count_by_bin = pd.Series(state_bins[idx_t]).value_counts()
    for b in all_bins:
        equal_rows.append({"time": t, "state_bin": b, "n_cells": int(count_by_bin.get(b, 0)), "total_time_cells": int(len(idx_t))})
equal_counts = pd.DataFrame(equal_rows)

# Fig 4.11a: original-depth vs equal-depth stacked bars.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=False)
orig_pivot = counts_by_state.pivot_table(index="time", columns="state_bin", values="n_cells", fill_value=0)
equal_pivot = equal_counts.pivot_table(index="time", columns="state_bin", values="n_cells", fill_value=0)
orig_pivot.plot(kind="bar", stacked=True, ax=axes[0], width=0.85, legend=False)
equal_pivot.plot(kind="bar", stacked=True, ax=axes[1], width=0.85, legend=False)
axes[0].set_title("Original observed depth")
axes[1].set_title(f"Equal-depth subsample n={n_min}")
for ax in axes:
    ax.set_xlabel("time label")
    ax.set_ylabel("observed/sample cells")
axes[1].legend(title="state bin", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("EB counts are sampling-depth diagnostics, not absolute abundance", y=1.02)
save_figure(fig, "fig4_11a_eb_observed_counts.png")

# Depth-sweep bootstrap counts and log growth proxy.
depth_grid = [d for d in [100, 200, 400, n_min] if d <= n_min]
depth_grid = sorted(set(depth_grid))
bootstrap_count_rows = []
for depth in depth_grid:
    for rep in range(BOOTSTRAP_REPEATS):
        selected = []
        for t in unique_times:
            idx = np.flatnonzero(labels_all == t)
            selected.append(rng.choice(idx, size=depth, replace=False))
        selected = np.concatenate(selected)
        for t in unique_times:
            idx_t = selected[labels_all[selected] == t]
            count_by_bin = pd.Series(state_bins[idx_t]).value_counts()
            for b in all_bins:
                c = int(count_by_bin.get(b, 0))
                bootstrap_count_rows.append({
                    "depth": int(depth), "repeat": int(rep), "time": t, "state_bin": b,
                    "count": c, "proportion": float(c / int(depth)),
                })
boot_counts = pd.DataFrame(bootstrap_count_rows)

eps_growth = 0.5
raw_rows = []
for t0, t1 in zip(unique_times[:-1], unique_times[1:]):
    c0 = counts_by_state[counts_by_state["time"] == t0].set_index("state_bin")
    c1 = counts_by_state[counts_by_state["time"] == t1].set_index("state_bin")
    for b in all_bins:
        n0 = int(c0.loc[b, "n_cells"])
        n1 = int(c1.loc[b, "n_cells"])
        p0 = float(c0.loc[b, "proportion"])
        p1 = float(c1.loc[b, "proportion"])
        raw_rows.append({
            "time_bridge": f"{t0}->{t1}",
            "time_t": t0,
            "time_t_next": t1,
            "state_bin": b,
            "raw_count_t": n0,
            "raw_count_t_next": n1,
            "raw_count_growth_proxy": float(np.log((n1 + eps_growth) / (n0 + eps_growth))),
            "normalized_proportion_change": float(p1 - p0),
        })
raw_growth = pd.DataFrame(raw_rows)

boot_proxy_rows = []
for depth in depth_grid:
    for rep in range(BOOTSTRAP_REPEATS):
        sub = boot_counts[(boot_counts["depth"] == depth) & (boot_counts["repeat"] == rep)]
        for t0, t1 in zip(unique_times[:-1], unique_times[1:]):
            s0 = sub[sub["time"] == t0].set_index("state_bin")
            s1 = sub[sub["time"] == t1].set_index("state_bin")
            for b in all_bins:
                n0 = int(s0.loc[b, "count"])
                n1 = int(s1.loc[b, "count"])
                boot_proxy_rows.append({
                    "depth": int(depth),
                    "repeat": int(rep),
                    "time_bridge": f"{t0}->{t1}",
                    "state_bin": b,
                    "equal_depth_growth_proxy": float(np.log((n1 + eps_growth) / (n0 + eps_growth))),
                })
boot_proxy = pd.DataFrame(boot_proxy_rows)
proxy_summary_all_depths = boot_proxy.groupby(["depth", "time_bridge", "state_bin"]).agg(
    proxy_mean=("equal_depth_growth_proxy", "mean"),
    proxy_ci_low=("equal_depth_growth_proxy", lambda x: float(np.quantile(x, 0.025))),
    proxy_ci_high=("equal_depth_growth_proxy", lambda x: float(np.quantile(x, 0.975))),
).reset_index()
proxy_equal_depth = proxy_summary_all_depths[proxy_summary_all_depths["depth"] == n_min].rename(columns={
    "proxy_mean": "equal_depth_proxy_mean",
    "proxy_ci_low": "equal_depth_proxy_ci_low",
    "proxy_ci_high": "equal_depth_proxy_ci_high",
})

downsampling_table = raw_growth.merge(
    proxy_equal_depth[["time_bridge", "state_bin", "equal_depth_proxy_mean", "equal_depth_proxy_ci_low", "equal_depth_proxy_ci_high"]],
    on=["time_bridge", "state_bin"],
    how="left",
)
ci_width = downsampling_table["equal_depth_proxy_ci_high"] - downsampling_table["equal_depth_proxy_ci_low"]
inside_ci = (downsampling_table["raw_count_growth_proxy"] >= downsampling_table["equal_depth_proxy_ci_low"]) & (downsampling_table["raw_count_growth_proxy"] <= downsampling_table["equal_depth_proxy_ci_high"])
downsampling_table["stable_under_subsampling"] = np.where(inside_ci & (ci_width < 1.0), "stable", "sensitive")
downsampling_table["state_label_source"] = "k-means on standardized PC-20 because EB file has no state/cluster labels"
downsampling_table["abundance_claim_boundary"] = "raw counts reflect sampling depth; not absolute biological abundance"
save_csv(OUT_DIR / "table4_6_eb_downsampling_diagnostics.csv", downsampling_table)
save_csv(CACHE_DIR / "exp9_depth_sweep_growth_proxy.csv", proxy_summary_all_depths)

# Adjacent bridge OT sampled endpoint diagnostics.
# This is an OT sampled endpoint diagnostic, not a trained CFM bridge and not a growth/abundance model.
def bridge_sampling_diagnostic(time_a, time_b, sampling_setting: str, cap: int, seed: int):
    idx_a_all = np.flatnonzero(labels_all == time_a)
    idx_b_all = np.flatnonzero(labels_all == time_b)
    rng_local = np.random.default_rng(seed)
    if sampling_setting == "original_depth":
        n_source = min(len(idx_a_all), int(cap))
        n_target = min(len(idx_b_all), int(cap))
    elif sampling_setting == "equal_depth":
        n_source = n_target = min(len(idx_a_all), len(idx_b_all), int(cap))
    else:
        raise ValueError(f"unknown sampling_setting={sampling_setting}")
    ia = np.sort(rng_local.choice(idx_a_all, size=n_source, replace=False))
    ib = np.sort(rng_local.choice(idx_b_all, size=n_target, replace=False))
    Xa, Xb = pcs_all[ia], pcs_all[ib]
    Cb, scaleb = compute_cost_matrix(Xa, Xb, normalize=True)
    pib, infob = sinkhorn_plan(Cb, epsilon=SINKHORN_EPSILON, return_info=True)
    diagb = coupling_diagnostics(pib, Cb)
    n_sample = min(4096, max(1024, 4 * n_source))
    sampled_i, sampled_j = sample_from_plan(pib, n_sample, seed=seed + 17)
    sampled_endpoint = Xb[sampled_j]
    pred_bins = state_bins[ib][sampled_j]
    target_bins = state_bins[ib]
    pred_prop = pd.Series(pred_bins).value_counts(normalize=True)
    target_prop = pd.Series(target_bins).value_counts(normalize=True)
    bin_err = sum(abs(float(pred_prop.get(k, 0.0)) - float(target_prop.get(k, 0.0))) for k in all_bins)
    return {
        "time_bridge": f"{time_a}->{time_b}",
        "sampling_setting": sampling_setting,
        "n_source": int(n_source),
        "n_target": int(n_target),
        "endpoint_mmd_pc20": mmd_rbf(sampled_endpoint, Xb),
        "sliced_w2_pc20": sliced_w2(sampled_endpoint, Xb, seed=seed + 23),
        "state_bin_terminal_proportion_error": float(bin_err),
        "expected_cost_normalized": float(diagb["expected_cost"]),
        "effective_support": float(diagb["effective_support"]),
        "sinkhorn_converged": bool(infob["sinkhorn_converged"]),
        "diagnostic_type": "ot_sampled_endpoint_diagnostic_not_trained_cfm",
        "claim_boundary": "standardized PC-20 OT sampled endpoint diagnostic; not trained CFM, not observed lineage, not growth or absolute abundance",
    }

bridge_cap = 120 if SMOKE_MODE else 800
bridge_rows = []
for bridge_id, (a, b) in enumerate(zip(unique_times[:-1], unique_times[1:])):
    bridge_rows.append(bridge_sampling_diagnostic(a, b, "original_depth", cap=bridge_cap, seed=410 + 10 * bridge_id))
    bridge_rows.append(bridge_sampling_diagnostic(a, b, "equal_depth", cap=min(bridge_cap, n_min), seed=411 + 10 * bridge_id))
bridge_sampling_table = pd.DataFrame(bridge_rows)
save_csv(OUT_DIR / "table4_6b_eb_bridge_sampling_diagnostics.csv", bridge_sampling_table)
save_csv(CACHE_DIR / "exp9_adjacent_bridge_ot_sampled_endpoint_diagnostics.csv", bridge_sampling_table)

boundary_rows = [
    {"assumption": "state labels", "status": "coarse k-means bins used", "claim_boundary": "diagnostic bins, not validated cell types"},
    {"assumption": "cell counts", "status": "observed destructive snapshot counts", "claim_boundary": "sampling depth, not absolute biological abundance"},
    {"assumption": "subsampling", "status": f"equal-depth bootstrap repeats={BOOTSTRAP_REPEATS}", "claim_boundary": "stability diagnostic only"},
    {"assumption": "adjacent bridge OT", "status": "original-depth and equal-depth OT sampled endpoint diagnostics in PC-20", "claim_boundary": "diagnostic only; not trained CFM and not observed lineage"},
]
boundary_table = pd.DataFrame(boundary_rows)
save_csv(OUT_DIR / "table4_7_biological_assumption_boundary.csv", boundary_table)

# Fig 4.11b: raw-count proxy vs equal-depth/depth-sweep bootstrap CI.
plot_df = downsampling_table.copy()
plot_df["label"] = plot_df["time_bridge"] + " | bin " + plot_df["state_bin"].astype(str)
plot_df = plot_df.sort_values(["time_bridge", "state_bin"]).reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
x = np.arange(len(plot_df))
y = plot_df["equal_depth_proxy_mean"].to_numpy()
yerr = np.vstack([
    y - plot_df["equal_depth_proxy_ci_low"].to_numpy(),
    plot_df["equal_depth_proxy_ci_high"].to_numpy() - y,
])
axes[0].errorbar(x, y, yerr=yerr, fmt="o", color=PALETTE["ot"], ecolor="0.65", elinewidth=1.0, capsize=2, label="equal-depth 95% CI")
axes[0].scatter(x, plot_df["raw_count_growth_proxy"], color=PALETTE["diagnostic"], s=18, label="raw-count proxy")
axes[0].axhline(0.0, color="0.35", linewidth=0.8)
axes[0].set_xticks(x[::max(1, len(x)//12)], plot_df["label"].iloc[::max(1, len(x)//12)], rotation=65, ha="right")
axes[0].set_ylabel("log growth proxy")
axes[0].set_title("Raw-count proxy versus equal-depth bootstrap")
axes[0].legend(frameon=False)

for depth, group in proxy_summary_all_depths.groupby("depth"):
    width = (group["proxy_ci_high"] - group["proxy_ci_low"]).mean()
    axes[1].scatter(depth, width, color=PALETTE["program"], s=40)
axes[1].plot(
    sorted(proxy_summary_all_depths["depth"].unique()),
    [float((proxy_summary_all_depths[proxy_summary_all_depths["depth"] == d]["proxy_ci_high"] - proxy_summary_all_depths[proxy_summary_all_depths["depth"] == d]["proxy_ci_low"]).mean()) for d in sorted(proxy_summary_all_depths["depth"].unique())],
    color=PALETTE["program"], linewidth=1.2,
)
axes[1].set_xlabel("equal depth per time")
axes[1].set_ylabel("mean bootstrap CI width")
axes[1].set_title("Depth-sweep uncertainty")
fig.suptitle("EB count proxy is a sampling-depth diagnostic, not absolute abundance", y=1.02)
save_figure(fig, "fig4_11b_sampling_depth_sensitivity.png")
downsampling_table.head()


## Exp 9b. WFR-FM Sampling-Depth Sensitivity

Load the internal WFR-FM implementation outputs from `scripts/run_ch04_wfrfm_sampling_sensitivity.py`.  By default this section uses the full uncapped run when the `_full` artifacts exist, while `CH04_WFRFM_OUTPUT_SUFFIX` can select another run.  The full run preserves high growth-rank agreement between raw observed depth and equal-depth controls, so the model has not lost the main state-space structure.  However, growth magnitude and part of the sign agreement remain sensitive to the mass convention supplied by snapshot sampling depth.  Therefore the inferred growth field should be treated as a model-dependent hypothesis rather than a direct proliferation/apoptosis rate without external abundance calibration, proliferation, apoptosis, or lineage evidence.

In [ ]:
from IPython.display import Image, display

def resolve_wfrfm_output_suffix() -> str:
    env_suffix = os.environ.get("CH04_WFRFM_OUTPUT_SUFFIX")
    if env_suffix is not None:
        return env_suffix.strip().lstrip("_")
    full_summary = OUT_DIR / "wfrfm_sampling_sensitivity_summary_full.json"
    return "full" if full_summary.exists() else ""

WFRFM_OUTPUT_SUFFIX = resolve_wfrfm_output_suffix()
def wfrfm_output_name(stem: str, ext: str) -> str:
    suffix = WFRFM_OUTPUT_SUFFIX.strip().lstrip("_")
    return f"{stem}_{suffix}.{ext}" if suffix else f"{stem}.{ext}"

wfrfm_growth_path = OUT_DIR / wfrfm_output_name("table4_6c_wfrfm_growth_by_bin", "csv")
wfrfm_sensitivity_path = OUT_DIR / wfrfm_output_name("table4_6d_wfrfm_sampling_sensitivity", "csv")
wfrfm_summary_path = OUT_DIR / wfrfm_output_name("wfrfm_sampling_sensitivity_summary", "json")
wfrfm_figure_path = FIG_DIR / wfrfm_output_name("fig4_11d_wfrfm_growth_sensitivity", "png")

required_wfrfm_outputs = [wfrfm_growth_path, wfrfm_sensitivity_path, wfrfm_summary_path, wfrfm_figure_path]
missing_wfrfm = [str(path) for path in required_wfrfm_outputs if not path.exists()]
if missing_wfrfm:
    raise FileNotFoundError("Run scripts/run_ch04_wfrfm_sampling_sensitivity.py before this section; missing: " + ", ".join(missing_wfrfm))

wfrfm_growth_by_bin = pd.read_csv(wfrfm_growth_path)
wfrfm_sampling_sensitivity = pd.read_csv(wfrfm_sensitivity_path)
wfrfm_summary = json.loads(wfrfm_summary_path.read_text())

if WFRFM_OUTPUT_SUFFIX == "full":
    assert wfrfm_summary.get("smoke") is False, "Full WFR-FM output must not be smoke mode."
    assert int(wfrfm_summary.get("epochs")) == 300, "Full WFR-FM output must use 300 epochs."
    assert wfrfm_summary.get("raw_observed_depth_was_capped") is False, "Full WFR-FM raw observed depth must be uncapped."
    assert wfrfm_summary.get("external_baseline_runtime_dependency") is False, "Exp 9b must use the internal WFR-FM implementation."

print("WFR-FM sampling sensitivity summary:")
print({
    "implementation": wfrfm_summary.get("implementation"),
    "external_baseline_runtime_dependency": wfrfm_summary.get("external_baseline_runtime_dependency"),
    "output_suffix": wfrfm_summary.get("output_suffix"),
    "dim": wfrfm_summary.get("dim"),
    "epochs": wfrfm_summary.get("epochs"),
    "batch_size": wfrfm_summary.get("batch_size"),
    "equal_seeds": wfrfm_summary.get("equal_seeds"),
    "raw_observed_depth_was_capped": wfrfm_summary.get("raw_observed_depth_was_capped"),
    "runtime_sec": round(float(wfrfm_summary.get("runtime_sec", 0.0)), 2),
})
print("implementation note:", wfrfm_summary.get("internal_implementation_note"))
print("sampling caveat:", wfrfm_summary.get("sampling_depth_caveat"))
display(wfrfm_sampling_sensitivity.head())
display(Image(filename=str(wfrfm_figure_path)))

## Exp 10. Stochastic Bridge Demo

A synthetic 2D demonstration comparing deterministic normalized bridge paths with stochastic bridge paths.  Each panel records normalized mass = 1 and is not interpreted as EB growth, death, proliferation, or abundance.


In [ ]:
def synthetic_bridge_samples(n=900, seed=42):
    rng = np.random.default_rng(seed)
    x0 = rng.normal(loc=[-1.5, 0.0], scale=[0.25, 0.45], size=(n, 2))
    mix = rng.uniform(size=n) < 0.5
    x1 = np.empty((n, 2))
    x1[mix] = rng.normal(loc=[1.4, 0.8], scale=[0.28, 0.25], size=(mix.sum(), 2))
    x1[~mix] = rng.normal(loc=[1.4, -0.8], scale=[0.28, 0.25], size=((~mix).sum(), 2))
    return x0.astype(np.float32), x1.astype(np.float32)

x0_syn, x1_syn = synthetic_bridge_samples()
t_grid_demo = [0, 0.25, 0.5, 0.75, 1]
rng = np.random.default_rng(401)
fig, axes = plt.subplots(2, len(t_grid_demo), figsize=(14, 5.2), sharex=True, sharey=True)
for col, tval in enumerate(t_grid_demo):
    det = (1 - tval) * x0_syn + tval * x1_syn
    noise_scale = 0.35 * math.sqrt(max(tval * (1 - tval), 0.0))
    stoch = det + rng.normal(scale=noise_scale, size=det.shape)
    for row, pts, label in [(0, det, "deterministic"), (1, stoch, "stochastic")]:
        ax = axes[row, col]
        ax.scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.35, linewidths=0, color=PALETTE["source"] if row == 0 else PALETTE["diagnostic"])
        ax.set_title(f"{label}\nt={tval:g}, mass=1")
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle("Synthetic normalized stochastic bridge demo; not EB abundance or growth", y=1.02)
save_figure(fig, "fig4_11c_stochastic_bridge_demo.png")
save_json(CACHE_DIR / "exp10_stochastic_bridge_manifest.json", {"normalized_mass_per_panel": 1.0, "claim_boundary": "not interpreted as proliferation/death"})


## Exp 11. Prior Boundary Audit

A claim-boundary table for common biological priors.  This is a table-only audit; any prior would require external evidence before upgrading a diagnostic into a biological claim.


In [ ]:
prior_rows = [
    {
        "prior": "RNA velocity alignment",
        "prior_type": "directional transcriptomic prior",
        "injection_point": "loss / diagnostic",
        "external_evidence_required": "validated velocity preprocessing, model assumptions, perturbation or lineage corroboration",
        "allowed_claim": "flow is more aligned with a specified RNA-velocity vector field under stated preprocessing",
        "forbidden_claim": "learned coupling is observed lineage or validated developmental fate",
    },
    {
        "prior": "marker monotonicity",
        "prior_type": "marker trend constraint",
        "injection_point": "loss / readout / diagnostic",
        "external_evidence_required": "curated marker set and expected direction from independent biology",
        "allowed_claim": "generated trajectories respect the specified marker monotonicity prior",
        "forbidden_claim": "marker prior proves true temporal ordering without external validation",
    },
    {
        "prior": "cell-type labels",
        "prior_type": "categorical annotation prior",
        "injection_point": "sampling / readout / diagnostic",
        "external_evidence_required": "validated annotation protocol and uncertainty handling",
        "allowed_claim": "terminal label proportions or forbidden transitions under provided labels",
        "forbidden_claim": "labels create same-cell lineage observations",
    },
    {
        "prior": "lineage barcodes",
        "prior_type": "external lineage evidence",
        "injection_point": "loss / sampling / evaluation",
        "external_evidence_required": "barcode assay with clone calling and sampling bias model",
        "allowed_claim": "agreement with barcode-derived clone constraints",
        "forbidden_claim": "unobserved transitions are recovered without barcode coverage",
    },
    {
        "prior": "GRN constraints",
        "prior_type": "mechanistic regulatory prior",
        "injection_point": "loss / diagnostic",
        "external_evidence_required": "GRN source, confidence scores, cell-context validation",
        "allowed_claim": "flow is consistent with selected GRN constraints under the chosen model",
        "forbidden_claim": "GRN-constrained flow proves causal regulation by itself",
    },
]
prior_audit = pd.DataFrame(prior_rows)
save_csv(OUT_DIR / "tableA_4_3_prior_boundary_audit.csv", prior_audit)
prior_audit


In [ ]:
# Optional small toy marker monotonicity sanity check.
lambdas = [0, 0.1, 1.0]
marker_rows = []
for lam in lambdas:
    vals = np.linspace(0, 1, 50) + np.random.default_rng(int(lam * 100 + 5)).normal(scale=0.08 / (1 + lam), size=50)
    violations = np.maximum(0, -np.diff(vals)).sum()
    marker_rows.append({"lambda": lam, "monotonicity_violation_sum": float(violations)})
marker_table = pd.DataFrame(marker_rows)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(marker_table["lambda"], marker_table["monotonicity_violation_sum"], marker="o")
ax.set_xlabel("lambda")
ax.set_ylabel("toy marker violation")
ax.set_title("Toy prior-strength sanity check")
save_figure(fig, "figA_4_1_prior_strength_sanity_check.png")
marker_table


## Artifact Manifest

This final section verifies the figures, tables, JSON files, and split-specific manifest generated by this notebook.


In [ ]:
expected_figures = [
    "fig4_11a_eb_observed_counts.png",
    "fig4_11b_sampling_depth_sensitivity.png",
    wfrfm_output_name("fig4_11d_wfrfm_growth_sensitivity", "png"),
    "fig4_11c_stochastic_bridge_demo.png",
    "figA_4_1_prior_strength_sanity_check.png",
]
expected_tables = [
    "eb_data_summary.json",
    "table4_6_eb_downsampling_diagnostics.csv",
    "table4_7_biological_assumption_boundary.csv",
    "table4_6b_eb_bridge_sampling_diagnostics.csv",
    wfrfm_output_name("table4_6c_wfrfm_growth_by_bin", "csv"),
    wfrfm_output_name("table4_6d_wfrfm_sampling_sensitivity", "csv"),
    wfrfm_output_name("wfrfm_sampling_sensitivity_summary", "json"),
    "tableA_4_3_prior_boundary_audit.csv",
    "claim_boundary_checklist.csv",
]

run_config = {
    "SMOKE_MODE": bool(SMOKE_MODE),
    "TRAINING_STEPS": int(TRAINING_STEPS),
    "BATCH_SIZE": int(BATCH_SIZE),
    "DEFAULT_NFE": int(DEFAULT_NFE),
    "DEVICE": str(DEVICE),
}
save_json(OUT_DIR / "run_config_04_3_sampling_depth_and_claim_boundaries.json", run_config)

claim_boundary_items = [
    ("training space marked", True),
    ("cost space marked", True),
    ("display space marked", True),
    ("readout space marked where applicable", True),
    ("no PHATE distance used as EB OT cost except Exp 7 contrastive diagnostic", True),
    ("no EB model trained in PHATE", True),
    ("no EB distributional metric computed in PHATE", True),
    ("no toy fate claim generalized to EB", True),
    ("no normalized snapshot proportion treated as absolute abundance", True),
    ("no straightness treated as biological validation", True),
    ("no coupling treated as observed lineage", True),
    ("no raw expected cost compared across representations", True),
    ("off-manifold reference set recorded in Chapter 4.2", True),
    ("reflow endpoint metrics computed against real target", True),
    ("stochastic bridge not interpreted as proliferation/death", True),
]
checklist_table = pd.DataFrame([{"item": item, "status": "pass" if ok else "fail"} for item, ok in claim_boundary_items])
save_csv(OUT_DIR / "claim_boundary_checklist.csv", checklist_table)

manifest_rows = []
for key, value in run_config.items():
    manifest_rows.append({"artifact": f"RUN_CONFIG:{key}={value}", "kind": "run_config", "exists": True, "bytes": 0})
for name in expected_figures:
    p = FIG_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "figure", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
for name in expected_tables:
    p = OUT_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "table_or_json", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
artifact_manifest = pd.DataFrame(manifest_rows)
save_csv(OUT_DIR / "artifact_manifest_04_3_sampling_depth_and_claim_boundaries.csv", artifact_manifest)

missing = artifact_manifest.loc[(artifact_manifest["kind"] != "run_config") & (~artifact_manifest["exists"])]
if not missing.empty:
    raise FileNotFoundError("Missing expected Chapter 4 split artifacts: " + ", ".join(missing["artifact"].tolist()))

print("Artifact manifest")
display(artifact_manifest)
